In [1]:
# ============================================================
# Nepal Election 2082 – Full Candidate Vote Dataset
# Source: result.election.gov.np  (Election Commission of Nepal)
# ============================================================
import json
import time
from pathlib import Path

import pandas as pd
import requests

BASE = "https://result.election.gov.np"
HANDLER = "/Handlers/SecureJson.ashx?file=JSONFiles/Election2082"
OUTPUT_CSV = "nepal_election_2082_results_ec.csv"
OUTPUT_JSON = "nepal_election_2082_results_ec.json"
LOCAL_FALLBACK_JSON = "nepal_election_2082_results.json"
LOCAL_FALLBACK_CSV = "nepal_election_2082_results.csv"
PROJECT_DIR = Path.cwd().resolve()
OUTPUT_CSV_PATH = PROJECT_DIR / OUTPUT_CSV
OUTPUT_JSON_PATH = PROJECT_DIR / OUTPUT_JSON
LOCAL_FALLBACK_JSON_PATH = PROJECT_DIR / LOCAL_FALLBACK_JSON
LOCAL_FALLBACK_CSV_PATH = PROJECT_DIR / LOCAL_FALLBACK_CSV

USE_LIVE_FETCH = False  # Set to True to force fresh download from EC API

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json, text/plain, */*",
})


def init_session(retries: int = 2) -> None:
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            SESSION.get(BASE + "/", timeout=12)
            csrf = SESSION.cookies.get("CsrfToken") or SESSION.cookies.get("csrfToken") or ""
            SESSION.headers.update({
                "Referer": BASE + "/",
                "X-CSRF-Token": csrf,
                "RequestVerificationToken": csrf,
            })
            return
        except Exception as exc:
            last_error = exc
            if attempt < retries:
                time.sleep(0.4 * attempt)
    raise last_error


def fetch_json(path: str, retries: int = 2):
    url = BASE + HANDLER + path
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            resp = SESSION.get(url, timeout=12)
            if resp.status_code == 403 and attempt < retries:
                init_session()
                continue
            resp.raise_for_status()
            return json.loads(resp.content.decode("utf-8-sig"))
        except Exception as exc:
            last_error = exc
            if attempt < retries:
                time.sleep(0.4 * attempt)
                init_session()
    raise last_error


def scrape_live_ec_data():
    init_session()

    states = fetch_json("/Local/Lookup/states.json")
    districts = fetch_json("/Local/Lookup/districts.json")
    consts_map = fetch_json("/HOR/Lookup/constituencies.json")

    state_id2name = {s["id"]: s["name"] for s in states}
    dist_id2name = {d["id"]: d["name"] for d in districts}
    dist_id2state = {d["id"]: d["parentId"] for d in districts}
    const_count = {c["distId"]: int(c["consts"]) for c in consts_map}

    all_rows = []
    errors = []

    for dist_id, n_consts in sorted(const_count.items()):
        state_id = dist_id2state.get(dist_id, "")
        state_name = state_id2name.get(state_id, "")
        dist_name = dist_id2name.get(dist_id, str(dist_id))

        for c_no in range(1, n_consts + 1):
            path = f"/HOR/FPTP/HOR-{dist_id}-{c_no}.json"
            try:
                candidates = fetch_json(path)
            except Exception as e:
                errors.append({"distId": dist_id, "const": c_no, "error": str(e)})
                continue

            for cand in candidates:
                all_rows.append({
                    "state_id": state_id,
                    "state_name": state_name,
                    "district_id": dist_id,
                    "district_name": dist_name,
                    "constituency_no": c_no,
                    "constituency_id": cand.get("SCConstID"),
                    "candidate_id": cand.get("CandidateID"),
                    "candidate_name": cand.get("CandidateName"),
                    "serial_no": cand.get("SerialNo"),
                    "gender": cand.get("Gender"),
                    "age": cand.get("Age"),
                    "dob": cand.get("DOB"),
                    "father_name": cand.get("FATHER_NAME"),
                    "spouse_name": cand.get("SPOUCE_NAME"),
                    "address": cand.get("ADDRESS"),
                    "ctz_district": cand.get("CTZDIST"),
                    "qualification": cand.get("QUALIFICATION"),
                    "institution": cand.get("NAMEOFINST"),
                    "experience": cand.get("EXPERIENCE"),
                    "other_details": cand.get("OTHERDETAILS"),
                    "party_id": cand.get("PartyID"),
                    "party_name": cand.get("PoliticalPartyName"),
                    "symbol_id": cand.get("SymbolID"),
                    "symbol_name": cand.get("SymbolName"),
                    "votes_received": cand.get("TotalVoteReceived"),
                    "votes_casted": cand.get("CastedVote"),
                    "total_voters": cand.get("TotalVoters"),
                    "rank": cand.get("Rank"),
                    "result": cand.get("Remarks"),
                    "is_elected": cand.get("Remarks") == "Elected",
                    "samudaya": cand.get("Samudaya"),
                })

            time.sleep(0.08)

    df_live = pd.DataFrame(all_rows)
    return df_live, errors


def load_local_fallback() -> pd.DataFrame:
    json_path = LOCAL_FALLBACK_JSON_PATH
    csv_path = LOCAL_FALLBACK_CSV_PATH

    if json_path.exists():
        try:
            return pd.read_json(json_path)
        except ValueError:
            with json_path.open("r", encoding="utf-8") as f:
                return pd.DataFrame(json.load(f))

    if csv_path.exists():
        return pd.read_csv(csv_path)

    raise FileNotFoundError("No fallback file found.")


# Try live EC fetch only when enabled; fallback to local cached data
errors = []
source_used = "Local fallback file"
if USE_LIVE_FETCH:
    source_used = "Live EC API"
    try:
        df = scrape_live_ec_data()
        if isinstance(df, tuple):
            df, errors = df
        if df.empty:
            raise RuntimeError("Live fetch returned no rows.")
    except Exception as exc:
        source_used = "Local fallback file"
        df = load_local_fallback()
        errors.append({"error": f"Live fetch failed: {exc}"})
else:
    df = load_local_fallback()

# Type cleanup for common numeric columns
for col in ["votes_received", "votes_casted", "total_voters", "age", "rank"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Ensure is_elected is available even in fallback data
if "is_elected" not in df.columns:
    if "result" in df.columns:
        df["is_elected"] = df["result"].astype(str).str.lower().eq("elected")
    elif "remarks" in df.columns:
        df["is_elected"] = df["remarks"].astype(str).str.lower().eq("elected")
    else:
        df["is_elected"] = False

# Save outputs
df.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")
df.to_json(OUTPUT_JSON_PATH, orient="records", force_ascii=False, indent=2)

print(f"Source used: {source_used}")
print(f"Dataset ready: {len(df):,} rows x {len(df.columns)} columns")
if set(["district_id", "constituency_no"]).issubset(df.columns):
    print(f"Constituencies covered: {df.groupby(['district_id', 'constituency_no']).ngroups}")
print(f"Elected candidates: {int(df['is_elected'].sum())}")
if errors:
    print(f"Warnings: {len(errors)}")
print(f"Saved CSV: {OUTPUT_CSV_PATH}")
print(f"Saved JSON: {OUTPUT_JSON_PATH}")

df.head()




Source used: Local fallback file
Dataset ready: 3,406 rows x 28 columns
Elected candidates: 0
Saved CSV: /Users/bikki/Desktop/Election82_data_analysis/nepal_election_2082_results_ec.csv
Saved JSON: /Users/bikki/Desktop/Election82_data_analysis/nepal_election_2082_results_ec.json


,election_id,election_type,state_id,state_name,district_id,district_name_en,district_name_np,area_id,area_name_en,area_name_np,...,candidate_name_en,candidate_name_np,candidate_slug,candidate_image_url,party_id,party_name_np,votes,winner,leading,is_elected
0,2082,HOR,1,Koshi Province,1,Taplejung,ताप्लेजुङ,1,Area 1,क्षेत्र नं. १,...,Kshitij Thebe,क्षितिज थेबे,kshitij-thebe-a1,https://file-r2.hamropatro.com/election-images...,105,नेपाल कम्युनिष्ट पार्टी (एकीकृत मार्क्सवादी ले...,13962,True,False,False
1,2082,HOR,1,Koshi Province,1,Taplejung,ताप्लेजुङ,1,Area 1,क्षेत्र नं. १,...,Gajendra Prasad Tumyang Limbu,गजेन्द्र प्रसाद तुम्याङ लिम्बु,gajendra-prasad-tumyang-limbu-a1,https://file-r2.hamropatro.com/election-images...,107,नेपाली काँग्रेस,11711,False,False,False
2,2082,HOR,1,Koshi Province,1,Taplejung,ताप्लेजुङ,1,Area 1,क्षेत्र नं. १,...,Khel Prasad Budakshetri,खेल प्रसाद बुडाक्षेत्री,khel-prasad-budakshetri-a1,https://file-r2.hamropatro.com/election-images...,108,नेपाली कम्युनिष्ट पार्टी,6775,False,False,False
3,2082,HOR,1,Koshi Province,1,Taplejung,ताप्लेजुङ,1,Area 1,क्षेत्र नं. १,...,Santosh Rai,सन्तोष राई,santosh-rai-a1,https://file-r2.hamropatro.com/election-images...,116,श्रम संस्कृति पार्टी,5209,False,False,False
4,2082,HOR,1,Koshi Province,1,Taplejung,ताप्लेजुङ,1,Area 1,क्षेत्र नं. १,...,Birendra Shrestha,बिरेन्द्र श्रेष्ठ,birendra-shrestha-a1,https://file-r2.hamropatro.com/election-images...,110,राष्ट्रिय स्वतन्त्र पार्टी,4155,False,False,False
